# Lab 05: Binary Classification on Pima Indians Diabetes Dataset
Clean, modular notebook submission for Perceptron (from scratch), Logistic Regression (from scratch), and Gaussian Naive Bayes.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB


## Dataset Loading


In [ ]:
def load_diabetes_data():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    columns = ["Pregnancies", "Glucose", "BloodPressure",
               "SkinThickness", "Insulin", "BMI", "DiabetesPedigreeFunction",
               "Age", "Outcome"]
    data = pd.read_csv(url, header=None, names=columns)
    print(data.head())
    return data


data = load_diabetes_data()


## Preprocessing Utilities


In [ ]:
def split_features_target(data):
    X = data.iloc[:, :-1].to_numpy(dtype=float)
    y = data.iloc[:, -1].to_numpy(dtype=int)
    return X, y


def standardize_train_val(X_train, X_val):
    train_mean = X_train.mean(axis=0)
    train_std = X_train.std(axis=0)
    train_std = np.where(train_std == 0, 1.0, train_std)
    X_train_std = (X_train - train_mean) / train_std
    X_val_std = (X_val - train_mean) / train_std
    return X_train_std, X_val_std


X, y = split_features_target(data)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_std, X_val_std = standardize_train_val(X_train, X_val)

print('Train shape:', X_train_std.shape, 'Validation shape:', X_val_std.shape)


## Shared Helper Functions


In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def log_loss(y_true, y_pred):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def misclassification_error(y_true, y_pred_probs, threshold=0.5):
    y_pred_labels = (y_pred_probs >= threshold).astype(int)
    return np.mean(y_pred_labels != y_true)


def evaluate_classification(y_true, y_pred_labels):
    return {
        'confusion_matrix': confusion_matrix(y_true, y_pred_labels),
        'accuracy': accuracy_score(y_true, y_pred_labels),
        'precision': precision_score(y_true, y_pred_labels, zero_division=0),
        'recall': recall_score(y_true, y_pred_labels, zero_division=0),
        'f1': f1_score(y_true, y_pred_labels, zero_division=0),
    }


def plot_single_curve(values, xlabel, ylabel, title, label):
    plt.figure(figsize=(8, 5))
    plt.plot(values, label=label)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_two_curves(train_values, val_values, xlabel, ylabel, title, train_label, val_label):
    plt.figure(figsize=(8, 5))
    plt.plot(train_values, label=train_label)
    plt.plot(val_values, label=val_label)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def print_metrics(title, metrics):
    print(f"\n{title}")
    print('Confusion Matrix:')
    print(metrics['confusion_matrix'])
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall   : {metrics['recall']:.4f}")
    print(f"F1 Score : {metrics['f1']:.4f}")


## Perceptron (From Scratch)
Binary Perceptron with sample-wise updates and per-iteration misclassification tracking.


In [ ]:
class PerceptronFromScratch:
    def __init__(self, learning_rate=0.01, n_iters=1000, random_state=42):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.random_state = random_state
        self.weights_ = None
        self.bias_ = None
        self.train_misclassification_history_ = []
        self.val_misclassification_history_ = []

    def decision_function(self, X):
        return np.dot(X, self.weights_) + self.bias_

    def predict(self, X):
        scores = self.decision_function(X)
        return (scores >= 0.0).astype(int)

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        n_samples, n_features = X_train.shape
        rng = np.random.default_rng(self.random_state)

        self.weights_ = rng.normal(loc=0.0, scale=0.01, size=n_features)
        self.bias_ = 0.0
        self.train_misclassification_history_ = []
        self.val_misclassification_history_ = []

        y_train_pm = np.where(y_train == 1, 1, -1)

        for _ in range(self.n_iters):
            for i in range(n_samples):
                x_i = X_train[i]
                y_i = y_train_pm[i]
                y_hat = 1 if (np.dot(x_i, self.weights_) + self.bias_) >= 0.0 else -1
                if y_hat != y_i:
                    update = self.learning_rate * y_i
                    self.weights_ += update * x_i
                    self.bias_ += update

            train_probs_like = self.predict(X_train).astype(float)
            self.train_misclassification_history_.append(
                misclassification_error(y_train, train_probs_like)
            )

            if X_val is not None and y_val is not None:
                val_probs_like = self.predict(X_val).astype(float)
                self.val_misclassification_history_.append(
                    misclassification_error(y_val, val_probs_like)
                )

        return self


In [ ]:
perceptron = PerceptronFromScratch(learning_rate=0.01, n_iters=100, random_state=42)
perceptron.fit(X_train_std, y_train, X_val_std, y_val)

perceptron_val_pred = perceptron.predict(X_val_std)
perceptron_metrics = evaluate_classification(y_val, perceptron_val_pred)
print_metrics('Perceptron (From Scratch) - Validation Metrics', perceptron_metrics)


In [ ]:
plot_two_curves(
    perceptron.train_misclassification_history_,
    perceptron.val_misclassification_history_,
    xlabel='Iteration',
    ylabel='Misclassification Rate',
    title='Perceptron: Misclassification Rate vs Iterations',
    train_label='Train Misclassification',
    val_label='Validation Misclassification',
)


## Logistic Regression (From Scratch)
Vectorized gradient descent with a fixed learning rate and only one loop over iterations.


In [ ]:
class LogisticRegressionFromScratch:
    def __init__(self, learning_rate=0.01, n_iters=1000, random_state=42):
        self.learning_rate = learning_rate
        self.n_iters = n_iters
        self.random_state = random_state
        self.weights_ = None
        self.bias_ = None
        self.train_misclassification_history_ = []
        self.val_misclassification_history_ = []
        self.train_log_loss_history_ = []
        self.val_log_loss_history_ = []

    def predict_proba(self, X):
        return sigmoid(np.dot(X, self.weights_) + self.bias_)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        n_samples, n_features = X_train.shape
        rng = np.random.default_rng(self.random_state)

        self.weights_ = rng.normal(loc=0.0, scale=0.01, size=n_features)
        self.bias_ = 0.0

        self.train_misclassification_history_ = []
        self.val_misclassification_history_ = []
        self.train_log_loss_history_ = []
        self.val_log_loss_history_ = []

        for _ in range(self.n_iters):
            p = sigmoid(X_train @ self.weights_ + self.bias_)
            err = p - y_train
            self.weights_ -= self.learning_rate * ((X_train.T @ err) / n_samples)
            self.bias_ -= self.learning_rate * np.mean(err)
            p_train = sigmoid(X_train @ self.weights_ + self.bias_)
            self.train_misclassification_history_.append(misclassification_error(y_train, p_train))
            self.train_log_loss_history_.append(log_loss(y_train, p_train))
            if X_val is not None and y_val is not None:
                p_val = sigmoid(X_val @ self.weights_ + self.bias_)
                self.val_misclassification_history_.append(misclassification_error(y_val, p_val))
                self.val_log_loss_history_.append(log_loss(y_val, p_val))

        return self


In [ ]:
log_reg = LogisticRegressionFromScratch(learning_rate=0.01, n_iters=1000, random_state=42)
log_reg.fit(X_train_std, y_train, X_val_std, y_val)

log_reg_val_pred = log_reg.predict(X_val_std)
log_reg_metrics = evaluate_classification(y_val, log_reg_val_pred)
print_metrics('Logistic Regression (From Scratch) - Validation Metrics', log_reg_metrics)


In [ ]:
plot_two_curves(
    log_reg.train_misclassification_history_,
    log_reg.val_misclassification_history_,
    xlabel='Iteration',
    ylabel='Misclassification Rate',
    title='Logistic Regression: Misclassification Rate vs Iterations',
    train_label='Train Misclassification',
    val_label='Validation Misclassification',
)

plot_two_curves(
    log_reg.train_log_loss_history_,
    log_reg.val_log_loss_history_,
    xlabel='Iteration',
    ylabel='Log Loss',
    title='Logistic Regression: Log Loss vs Iterations',
    train_label='Train Log Loss',
    val_label='Validation Log Loss',
)


## Naive Bayes (GaussianNB)
Library-based Gaussian Naive Bayes for binary classification.


In [ ]:
naive_bayes = GaussianNB()


In [ ]:
naive_bayes.fit(X_train_std, y_train)
nb_val_pred = naive_bayes.predict(X_val_std).astype(int)
nb_val_probs = naive_bayes.predict_proba(X_val_std)[:, 1]

naive_bayes_metrics = evaluate_classification(y_val, nb_val_pred)
print_metrics('Gaussian Naive Bayes - Validation Metrics', naive_bayes_metrics)


## Final Comparison


In [ ]:
summary_df = pd.DataFrame(
    {
        'Perceptron (From Scratch)': {
            'Accuracy': perceptron_metrics['accuracy'],
            'Precision': perceptron_metrics['precision'],
            'Recall': perceptron_metrics['recall'],
            'F1': perceptron_metrics['f1'],
        },
        'Logistic Regression (From Scratch)': {
            'Accuracy': log_reg_metrics['accuracy'],
            'Precision': log_reg_metrics['precision'],
            'Recall': log_reg_metrics['recall'],
            'F1': log_reg_metrics['f1'],
        },
        'Gaussian Naive Bayes': {
            'Accuracy': naive_bayes_metrics['accuracy'],
            'Precision': naive_bayes_metrics['precision'],
            'Recall': naive_bayes_metrics['recall'],
            'F1': naive_bayes_metrics['f1'],
        },
    }
).T

summary_df


## Final Note
Rename this notebook before submission using: **`<roll>_<name>.ipynb`**
